# Set 06 – L1- und L2-Regularisierung im direkten Vergleich

Regularisierung ergänzt den Datenfehler um eine Strafe für große Koeffizienten.

- Ridge verwendet L2 und verkleinert Koeffizienten kontinuierlich.
- Lasso verwendet L1 und kann Koeffizienten exakt auf null setzen.
- alpha steuert in scikit-learn die Stärke der Regularisierung.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. Datensatz mit wenigen wirklich informativen Merkmalen

Wir erzeugen 30 Merkmale, aber nur fünf tragen direkt zum Ziel bei. Das ist eine geeignete Situation, um sparsames Verhalten von Lasso sichtbar zu machen.

In [ ]:
X_array, y, echte_koeffizienten = make_regression(
    n_samples=60,
    n_features=30,
    n_informative=5,
    noise=35,
    coef=True,
    random_state=42,
)
merkmalsnamen = [f"merkmal_{index:02d}" for index in range(1, 31)]
X = pd.DataFrame(X_array, columns=merkmalsnamen)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=42
)

wahrheit = pd.DataFrame({
    "Merkmal": merkmalsnamen,
    "echter_Koeffizient": echte_koeffizienten,
})
display(wahrheit.sort_values("echter_Koeffizient", key=abs, ascending=False).head(10).round(2))

## 2. Warum Skalierung vor Regularisierung?

L1 und L2 bestrafen die Koeffizienten, nicht die ursprünglichen Einheiten. StandardScaler sorgt dafür, dass alle Merkmale fair auf einer vergleichbaren Skala behandelt werden.

In [ ]:
modelle = {
    "Ohne Regularisierung": LinearRegression(),
    "Ridge L2": Ridge(alpha=10.0),
    "Lasso L1": Lasso(alpha=5.0, max_iter=20000),
}

pipelines = {
    name: Pipeline([
        ("skalierung", StandardScaler()),
        ("modell", modell),
    ])
    for name, modell in modelle.items()
}

for pipeline in pipelines.values():
    pipeline.fit(X_train, y_train)

## 3. Trainings- und Testfehler vergleichen

Regularisierung darf den Trainingsfehler etwas erhöhen, wenn dadurch die Leistung auf neuen Daten stabiler wird.

In [ ]:
metriken = []
for name, pipeline in pipelines.items():
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)
    metriken.append({
        "Modell": name,
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Test_R²": r2_score(y_test, test_pred),
    })

vergleich = pd.DataFrame(metriken).set_index("Modell")
display(vergleich.round(3))

## 4. Koeffizienten nebeneinander

Da alle Modelle dieselbe Skalierung verwenden, sind die Koeffizienten direkt vergleichbar.

Ridge zieht viele Werte in Richtung null. Lasso setzt schwache Koeffizienten teilweise exakt auf null.

In [ ]:
koeffizienten = pd.DataFrame({"Merkmal": merkmalsnamen})
for name, pipeline in pipelines.items():
    koeffizienten[name] = pipeline.named_steps["modell"].coef_

display(koeffizienten.round(3))
print("Exakt null bei Ridge:", int(np.isclose(koeffizienten["Ridge L2"], 0, atol=1e-10).sum()))
print("Exakt null bei Lasso:", int(np.isclose(koeffizienten["Lasso L1"], 0, atol=1e-10).sum()))

In [ ]:
sortierung = koeffizienten["Ohne Regularisierung"].abs().sort_values().index
plot_daten = koeffizienten.loc[sortierung].set_index("Merkmal")

ax = plot_daten.plot.barh(figsize=(11, 12), width=0.75)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Koeffizient nach Standardisierung")
ax.set_title("Direkter Vergleich der Koeffizienten")
plt.tight_layout()
plt.show()

## 5. Betrag der Koeffizienten

Die absolute Darstellung macht sichtbar, welche Merkmale Lasso vollständig entfernt und wie stark Ridge die übrigen Gewichte verkleinert.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 10), sharey=True)
for ax, name, farbe in zip(
    axes,
    ["Ohne Regularisierung", "Ridge L2", "Lasso L1"],
    ["#4C78A8", "#F58518", "#54A24B"],
):
    daten = koeffizienten.set_index("Merkmal")[name].abs().sort_values()
    ax.barh(daten.index, daten.values, color=farbe)
    ax.set_title(name)
    ax.set_xlabel("absoluter Koeffizient")
plt.tight_layout()
plt.show()

## 6. Einfluss von alpha

Kleine alpha-Werte ähneln der unregularisierten Lösung. Große Werte erzwingen einfachere Modelle. Zu viel Regularisierung kann Underfitting verursachen.

In [ ]:
alpha_werte = np.logspace(-2, 2.5, 24)
pfad_ergebnisse = []

for alpha in alpha_werte:
    for name, modell in [
        ("Ridge", Ridge(alpha=alpha)),
        ("Lasso", Lasso(alpha=alpha, max_iter=30000)),
    ]:
        pipeline = Pipeline([
            ("skalierung", StandardScaler()),
            ("modell", modell),
        ])
        pipeline.fit(X_train, y_train)
        koef = pipeline.named_steps["modell"].coef_
        pfad_ergebnisse.append({
            "Modell": name,
            "alpha": alpha,
            "Test_RMSE": np.sqrt(mean_squared_error(y_test, pipeline.predict(X_test))),
            "L1_Norm": np.abs(koef).sum(),
            "Anzahl_ungleich_null": int((np.abs(koef) > 1e-8).sum()),
        })

pfad = pd.DataFrame(pfad_ergebnisse)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, gruppe in pfad.groupby("Modell"):
    axes[0].plot(gruppe["alpha"], gruppe["Test_RMSE"], marker="o", label=name)
    axes[1].plot(gruppe["alpha"], gruppe["L1_Norm"], marker="o", label=name)
    axes[2].plot(gruppe["alpha"], gruppe["Anzahl_ungleich_null"], marker="o", label=name)

for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("alpha")
    ax.legend()
axes[0].set_ylabel("Test-RMSE")
axes[0].set_title("Generalisierungsfehler")
axes[1].set_ylabel("Summe absoluter Koeffizienten")
axes[1].set_title("Koeffizienten werden kleiner")
axes[2].set_ylabel("aktive Koeffizienten")
axes[2].set_title("Lasso erzeugt Sparsity")
plt.tight_layout()
plt.show()

## Was lernen wir daraus?

- Ridge ist oft sinnvoll, wenn viele Merkmale etwas beitragen oder stark korreliert sind.
- Lasso kann ein sparsames, leichter interpretierbares Modell erzeugen.
- Ein Koeffizient von null bedeutet nur, dass dieses Modell das Merkmal bei diesem alpha nicht nutzt.
- alpha ist ein Hyperparameter und darf nicht anhand des finalen Testsatzes gewählt werden.
- Die passende Methode und Stärke werden mit Cross-Validation bestimmt.